In [ ]:
# --- repo bootstrap (Colab + local compatible) ---
import sys
from pathlib import Path

repo = Path.cwd()
if repo.name == "notebooks":
    repo = repo.parent

if not (repo / "src").exists():
    !git clone https://github.com/thinkthoughts/ion-transport-waveform-pipeline.git
    %cd ion-transport-waveform-pipeline
    repo = Path.cwd()

if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

print("Repo root:", repo)


# 02 — Transport Path

Convert fixed well positions into smooth transport trajectories.

```text
start well → smooth path x_c(t) → velocity/acceleration checks → transport profile
```

Notebook 01 showed that approximate wells can be positioned. This notebook turns those static positions into a time-dependent shuttling path.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.ion_transport_waveform.transport_path import minimum_jerk_path, linear_path

fig_dir = repo / "figures"
fig_dir.mkdir(exist_ok=True)
data_dir = repo / "data" / "simulation_outputs"
data_dir.mkdir(parents=True, exist_ok=True)


## 1. Define transport task

We shuttle an ion-centered potential well from \(-160\,\mu m\) to \(+160\,\mu m\). The minimum-jerk trajectory has zero velocity and acceleration at the endpoints, making it a useful baseline for low-excitation transport.


In [ ]:
x0 = -160e-6
x1 = 160e-6
duration = 20e-6
n_steps = 1200

t = np.linspace(0, duration, n_steps)
path_mj = minimum_jerk_path(t, x0=x0, x1=x1, duration=duration)
path_linear = linear_path(t, x0=x0, x1=x1, duration=duration)

print(f"transport distance: {(x1-x0)*1e6:.1f} µm")
print(f"duration: {duration*1e6:.1f} µs")
print(f"samples: {n_steps}")


## 2. Compare linear and minimum-jerk paths

Linear position interpolation is simple but has discontinuous velocity at endpoints. Minimum-jerk interpolation removes that endpoint discontinuity.


In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(t * 1e6, path_linear * 1e6, label="linear path")
plt.plot(t * 1e6, path_mj * 1e6, label="minimum-jerk path")
plt.xlabel("time (µs)")
plt.ylabel("well center x_c(t) (µm)")
plt.title("Transport path from -160 µm to +160 µm")
plt.legend()
plt.tight_layout()
plt.savefig(fig_dir / "02_transport_path_profiles.png", dpi=180)
plt.show()

print(f"saved: {fig_dir / '02_transport_path_profiles.png'}")


## 3. Velocity and acceleration profiles

Transport excitation depends strongly on how sharply the well moves. Velocity and acceleration give the first constraint checks before full ion-motion simulation.


In [ ]:
dt = t[1] - t[0]
v_mj = np.gradient(path_mj, dt)
a_mj = np.gradient(v_mj, dt)
v_linear = np.gradient(path_linear, dt)
a_linear = np.gradient(v_linear, dt)

print(f"minimum-jerk peak speed: {np.max(np.abs(v_mj)):.3e} m/s")
print(f"minimum-jerk peak acceleration: {np.max(np.abs(a_mj)):.3e} m/s²")


In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(t * 1e6, v_mj, label="minimum-jerk velocity")
plt.plot(t * 1e6, v_linear, label="linear velocity", alpha=0.75)
plt.xlabel("time (µs)")
plt.ylabel("velocity (m/s)")
plt.title("Transport velocity profile")
plt.legend()
plt.tight_layout()
plt.savefig(fig_dir / "02_transport_velocity_profiles.png", dpi=180)
plt.show()

print(f"saved: {fig_dir / '02_transport_velocity_profiles.png'}")


In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(t * 1e6, a_mj, label="minimum-jerk acceleration")
plt.plot(t * 1e6, a_linear, label="linear acceleration", alpha=0.75)
plt.xlabel("time (µs)")
plt.ylabel("acceleration (m/s²)")
plt.title("Transport acceleration profile")
plt.legend()
plt.tight_layout()
plt.savefig(fig_dir / "02_transport_acceleration_profiles.png", dpi=180)
plt.show()

print(f"saved: {fig_dir / '02_transport_acceleration_profiles.png'}")


## 4. Duration sweep

A transport path is not only a geometric curve; it is a time allocation. The same distance becomes easier or harder depending on transport duration.


In [ ]:
durations = np.array([5, 10, 20, 40, 80], dtype=float) * 1e-6
peak_speed = []
peak_accel = []

for T in durations:
    tt = np.linspace(0, T, n_steps)
    xx = minimum_jerk_path(tt, x0=x0, x1=x1, duration=T)
    vv = np.gradient(xx, tt[1] - tt[0])
    aa = np.gradient(vv, tt[1] - tt[0])
    peak_speed.append(np.max(np.abs(vv)))
    peak_accel.append(np.max(np.abs(aa)))

peak_speed = np.array(peak_speed)
peak_accel = np.array(peak_accel)

for T, vpk, apk in zip(durations, peak_speed, peak_accel):
    print(f"T={T*1e6:5.1f} µs | peak speed={vpk:9.3e} m/s | peak accel={apk:9.3e} m/s²")


In [ ]:
plt.figure(figsize=(7.5, 4.5))
plt.loglog(durations * 1e6, peak_speed, marker="o", label="peak speed")
plt.loglog(durations * 1e6, peak_accel, marker="s", label="peak acceleration")
plt.xlabel("transport duration (µs)")
plt.ylabel("peak kinematic scale")
plt.title("Duration sweep: faster transport increases kinematic demand")
plt.legend()
plt.tight_layout()
plt.savefig(fig_dir / "02_duration_sweep_kinematics.png", dpi=180)
plt.show()

print(f"saved: {fig_dir / '02_duration_sweep_kinematics.png'}")


## 5. Save path data

Notebook 03 will convert this smooth path into a sequence of voltage solutions: one voltage vector per timestep.


In [ ]:
np.savez(
    data_dir / "transport_path_02.npz",
    t=t,
    path_minimum_jerk=path_mj,
    path_linear=path_linear,
    velocity_minimum_jerk=v_mj,
    acceleration_minimum_jerk=a_mj,
    x0=x0,
    x1=x1,
    duration=duration,
)

print(f"saved: {data_dir / 'transport_path_02.npz'}")


## 6. Next notebook

`03_waveform_generation.ipynb` should map the smooth target path into electrode-voltage waveforms:

```text
x_c(t) → voltage vector v(t) → waveform continuity/bandwidth checks
```
